## 从参数查询的NER数据集里取label为QUERY的数据

In [19]:
import json

dataset = []
with open('param_query.json', 'r', encoding='utf-8') as file:
    data = json.load(file)
for _, item in enumerate(data):
    if _ >= 1000:
        break
    dataset.append({
        'input': item['text'],
        'label': 'QUERY'
    })
    

## 从参数控制的数据集里取label为CONTROL的数据

In [20]:
from datasets import load_dataset

data = load_dataset('json', data_files="/dev_data/zkyao/code/Weld-GPT_demo/sentence_classify/param_control_en/sample_output.json", split='train')

for _, item in enumerate(data):
    if _ >= 1000:
        break
    dataset.append({
        'input': item['input'],
        'label': 'CONTROL'
    })

## 从刚刚采样的RAG数据集里取label为RAG的数据

In [21]:
from datasets import load_dataset
data = load_dataset('json', data_files="rag_sample_output_text.json", split='train')

In [22]:
import re

count = 0
pattern = r'\d+\.\s+([^\n]+)\n'
for item in data:
    text = item['input']
    matches = re.findall(pattern, text)
    for match in matches:
        if count >= 1000:
            break
        count += 1
        dataset.append({
            'input': match,
            'label': 'RAG'
        })
    if count >= 1000:
        break

In [23]:
len(dataset)

3000

In [32]:
import random

label_groups = {}
for data in dataset:
    label = data['label']
    if label not in label_groups:
        label_groups[label] = []
    label_groups[label].append(data)
    
train_dataset, validate_dataset, test_dataset = [], [], []
for label, group in label_groups.items():
    train_dataset.extend(group[:800])
    validate_dataset.extend(group[800:900])
    test_dataset.extend(group[900:1000])
random.seed(42)
random.shuffle(train_dataset)
random.shuffle(validate_dataset)
random.shuffle(test_dataset)
len(train_dataset), len(validate_dataset), len(test_dataset)

(2400, 300, 300)

In [33]:
fw = open('train.json', 'w+')
for data in train_dataset:
    fw.write(json.dumps(data, ensure_ascii=False)+'\n')
fw.close()
fw = open('validate.json', 'w+')
for data in validate_dataset:
    fw.write(json.dumps(data, ensure_ascii=False)+'\n')
fw.close()
fw = open('test.json', 'w+')
for data in test_dataset:
    fw.write(json.dumps(data, ensure_ascii=False)+'\n')
fw.close()